<a href="https://colab.research.google.com/github/carlosgarcia81f-create/AnalizadorDeFiniquitos/blob/main/AnalizadorDeFiniquito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#@title 📊 Configuración de la Auditoría { run: "auto" }

nombre_archivo = "00_01_24_PavHidalgo.xlsm" #@param {type:"string"}
filas_a_saltar = 11 #@param {type:"slider", min:0, max:50, step:1}
umbral_exceso = 0.3 #@param {type:"number"}
nombre_hoja = "12" #@param {type:"string"}
porcentaje_pareto = 80 #@param {type:"slider", min:50, max:95, step:5}

print(f"Configuración cargada para: {nombre_archivo}")

import pandas as pd
import numpy as np

#---------------------- 1. Cargamos el archivo----------------------------------------------------------------------------------------------------
df = pd.read_excel(
        nombre_archivo,
        sheet_name = nombre_hoja,
        skiprows=filas_a_saltar
    )
#Ver listado de columnas en el archivo cargado. (Deben visualizarse todas las que nos dan un tipo de información)
print(df.columns)

df_finiquito = df[[
    'Clave', 'Concepto', 'Unidad','Precio Unitario/Costo', 'Cantidad total estimada', 'Importe Contratado', 'Importe total estimado','NIVEL']
    ]

#print('Las siguientes son las columnas que utilizaremos:')
#print(df_finiquito.columns)

#---------------------- 2. Renombramos columnas----------------------------------------------------------------------------------------------------
df_finiquito = df_finiquito.rename(columns={
    'Precio Unitario/Costo': 'PU',
    'Importe Contratado': 'Monto_Contratado',
    'Importe total estimado': 'Monto_Ejecutado'
})

#---------------------- 3. Identificamos Nombres de Partidas y Subpartidas basado en la columna NIVEL-----------------------------------------------
# Si la columna 'NIVEL' tiene un 1, es Partida Principal
df_finiquito.loc[df_finiquito['NIVEL'] == 1, 'Partida_Principal'] = df_finiquito['Concepto']

# Si la columna 'NIVEL' tiene un 2, es Subpartida
df_finiquito.loc[df_finiquito['NIVEL'] == 2, 'Subpartida'] = df_finiquito['Concepto']

#---------------------- 4. Aplicamos el relleno (Forward Fill)--------------------------------------------------------------------------------------
# 1. Rellenamos primero las Partidas Principales (Nivel 1)
df_finiquito['Partida_Principal'] = df_finiquito['Partida_Principal'].ffill()

# 2. Si una fila es una nueva Partida Principal (Nivel 1),
# forzamos que la Subpartida sea un texto vacío o "Sin Subpartida" para que no arrastre la anterior.
df_finiquito.loc[df_finiquito['NIVEL'] == 1, 'Subpartida'] = "N/A"
# Al haber puesto "N/A" en el inicio de cada partida, el ffill solo arrastrará dentro de su propia sección.
df_finiquito['Subpartida'] = df_finiquito['Subpartida'].ffill()

#------------------- 5. Limpieza -------------------------------------------------------------------------------------------------------------------
#1.Nos quedamos solo con los renglones que SON conceptos (los que no tienen 1 ni 2)
#Generalmente los conceptos no tienen marca en esa columna o tienen un 0
df_finiquito_auditoria = df_finiquito[df_finiquito['PU'].notna()].copy()

# 2. Borramos las filas que digan TOTAL, SUMA, SUBTOTAL en la columna Precio Unitario
# El parámetro 'case=False' ignora si está en mayúsculas o minúsculas
# Se añade una comprobación para asegurar que 'PU' es de tipo string antes de usar .str
if not df_finiquito_auditoria['PU'].empty:
    df_finiquito_auditoria = df_finiquito_auditoria[
        ~df_finiquito_auditoria['PU'].astype(str).str.contains('TOTAL|SUMA|SUBTOTAL|RESUMEN', case=False, na=False)
    ]

# NUEVA LIMPIEZA: Borramos filas donde 'Concepto' es nulo o 'N/A'
df_finiquito_auditoria = df_finiquito_auditoria[
    df_finiquito_auditoria['Concepto'].notna() &
    (df_finiquito_auditoria['Concepto'] != 'N/A')
].copy()

#---------------------- 6. Visualización del resultado-----------------------------------------------------------------------------------------------
df_finiquito_auditoria[['Partida_Principal', 'Subpartida', 'Clave', 'Concepto', 'Monto_Contratado','Monto_Ejecutado']]

#----------------------7. Calculamos la variación porcentual de cada concepto respecto de lo contratado----------------------------------------------
#Esto nos ayuda a ver cuales conceptos rebasaron más el importe contratado
porcentajeRespectoContrato = umbral_exceso # Usamos la variable de configuración
df_finiquito_auditoria['Variacion_Pct'] = (df_finiquito_auditoria['Monto_Ejecutado'] - df_finiquito_auditoria['Monto_Contratado']) / df_finiquito_auditoria['Monto_Contratado']
# Create a new column for the formatted percentage for display purposes
df_finiquito_auditoria['Variacion_Pct_%'] = df_finiquito_auditoria['Variacion_Pct'].apply(lambda x: f'{x:.2%}')
# Filtramos los que superan el porcentaje señalado (using the numeric Variacion_Pct)
excesos = df_finiquito_auditoria[df_finiquito_auditoria['Variacion_Pct'] > porcentajeRespectoContrato]

print(f"Se encontraron {len(excesos)} conceptos con un porcentaje de {porcentajeRespectoContrato*100}% superior respecto del porcentaje contratado")
display(excesos[['Clave', 'Partida_Principal', 'Subpartida', 'Concepto', 'Monto_Contratado','Monto_Ejecutado','Variacion_Pct_%']])

#-----------------------------------------------------------------------------------------------------------------------------------------------------
#---------------------- 8. RESUMEN EJECUTIVO (CORREGIDO) ---------------------------------------------------------------------------------------------
resumen_ejecutivo = df_finiquito_auditoria.groupby(['Partida_Principal', 'Subpartida']).agg({
    'Monto_Contratado': 'sum',
    'Monto_Ejecutado': 'sum'
}).reset_index()

# 1. Recalculate Diferencia_Absoluta
resumen_ejecutivo['Diferencia_Absoluta'] = resumen_ejecutivo['Monto_Ejecutado'] - resumen_ejecutivo['Monto_Contratado']

# 2. Initialize %_Variacion_Global with a default value (0)
# Ensure the column is explicitly float type from the start
resumen_ejecutivo['%_Variacion_Global'] = 0.0

# 3. Handle cases where Monto_Contratado is 0 and Diferencia_Absoluta is also 0 (already 0 by initialization)

# 4. Identify rows where Monto_Contratado is 0 but Diferencia_Absoluta is not 0
condition_contracted_zero_diff_not_zero = (
    (resumen_ejecutivo['Monto_Contratado'] == 0) &
    (resumen_ejecutivo['Diferencia_Absoluta'] != 0)
)
resumen_ejecutivo.loc[condition_contracted_zero_diff_not_zero, '%_Variacion_Global'] = 100.0

# 5. For all other rows (where Monto_Contratado is not 0), calculate normally
condition_monto_contratado_not_zero = (resumen_ejecutivo['Monto_Contratado'] != 0)
resumen_ejecutivo.loc[condition_monto_contratado_not_zero, '%_Variacion_Global'] = (
    (resumen_ejecutivo.loc[condition_monto_contratado_not_zero, 'Diferencia_Absoluta'] /
    resumen_ejecutivo.loc[condition_monto_contratado_not_zero, 'Monto_Contratado']) * 100
).astype(float) # Explicitly cast to float to prevent FutureWarning

# Ordenamos
resumen_ejecutivo = resumen_ejecutivo.sort_values(by='Diferencia_Absoluta', ascending=False)

# Visualización con formato
print("--- RESUMEN DE AUDITORÍA POR SUBPARTIDAS ---")
display(resumen_ejecutivo.style.format({
    'Monto_Contratado': '${:,.2f}',
    'Monto_Ejecutado': '${:,.2f}',
    'Diferencia_Absoluta': '${:,.2f}',
    '%_Variacion_Global': '{:.2f}%'
}).background_gradient(subset=['%_Variacion_Global'], cmap='YlOrRd'))
#-----------------------------------------------------------------------------------------------------------------------------------------------------------
#---------------------- 9. PLANEACIÓN DE INSPECCIÓN FÍSICA (ANÁLISIS DE PARETO CON CONFIGURACIÓN) ------------------------------------------

# 1. Ordenamos de mayor a menor importancia económica
df_plan_inspeccion = df_finiquito_auditoria.sort_values(by='Monto_Ejecutado', ascending=False).copy()

# 2. Calculamos el peso de cada concepto y su acumulado
total_finiquito = df_plan_inspeccion['Monto_Ejecutado'].sum()
df_plan_inspeccion['%_Peso'] = (df_plan_inspeccion['Monto_Ejecutado'] / total_finiquito) * 100
df_plan_inspeccion['%_Acumulado'] = df_plan_inspeccion['%_Peso'].cumsum()

# 3. Definimos quiénes son del Grupo A (Prioridad Alta) usando el porcentaje configurable
#    El grupo B será un 5% adicional, y el resto el Grupo C.
threshold_alta = porcentaje_pareto # Usamos la variable de configuración (ej. 90%)
threshold_media = threshold_alta + 5 # Por ejemplo, 95%

df_plan_inspeccion['Prioridad'] = df_plan_inspeccion['%_Acumulado'].apply(
    lambda x: f'ALTA (Grupo A - cubre el {threshold_alta}%)' if x <= threshold_alta
    else (f'MEDIA (Grupo B - cubre hasta el {threshold_media}%)' if x <= threshold_media else 'BAJA (Grupo C)')
)

# 4. Filtramos la lista para tu bitácora de campo (solo los de alta prioridad)
# Se añade una comprobación para evitar el AttributeError si el DataFrame está vacío
if not df_plan_inspeccion.empty:
    df_plan_inspeccion['Prioridad'] = df_plan_inspeccion['Prioridad'].astype(str)
    lista_campo = df_plan_inspeccion[df_plan_inspeccion['Prioridad'].str.startswith('ALTA')]
else:
    print("Advertencia: df_plan_inspeccion está vacío. No se pueden generar elementos para la lista de campo.")
    lista_campo = pd.DataFrame(columns=['Partida_Principal', 'Subpartida', 'Clave', 'Concepto', 'Monto_Ejecutado', '%_Peso', '%_Acumulado', 'Prioridad']) # Crear un DataFrame vacío con las columnas esperadas

print(f"\n--- ESTRATEGIA DE INSPECCIÓN FÍSICA (ANÁLISIS DE PARETO {threshold_alta}/{100-threshold_alta}) ---")
print(f"Total de conceptos en la obra: {len(df_plan_inspeccion)}")
print(f"Conceptos críticos a revisar para cubrir el {threshold_alta}% del monto: {len(lista_campo)}")
print("-" * 50)

# Visualización para llevar a campo
display(lista_campo[['Partida_Principal', 'Subpartida', 'Clave', 'Concepto', 'Cantidad total estimada','Monto_Ejecutado', '%_Peso','%_Acumulado']].style.format({
    '%_Peso': '{:.2f}%',
    'Monto_Ejecutado': '${:,.2f}',
    'Cantidad total estimada': '{:,.0f}',
    '%_Acumulado': '{:.2f}%'
}).bar(subset=['%_Acumulado'], color='#5fba7d'))

print("\n--- RESUMEN COMPLETO DE PRIORIDADES ---")
# Filtramos los conceptos con Monto_Ejecutado > 0 (y por lo tanto %_Peso > 0)
df_plan_inspeccion_filtrado = df_plan_inspeccion[df_plan_inspeccion['Monto_Ejecutado'] > 0].copy()
display(df_plan_inspeccion_filtrado[['Prioridad','Partida_Principal', 'Subpartida', 'Clave', 'Concepto','Cantidad total estimada', 'Monto_Ejecutado', '%_Peso', '%_Acumulado']].style.format({
    '%_Peso': '{:.2f}%',
    'Monto_Ejecutado': '${:,.2f}',
    'Cantidad total estimada': '{:,.0f}',
    '%_Acumulado': '{:.2f}%'
}).background_gradient(subset=['%_Acumulado'], cmap='Blues'))

Configuración cargada para: 00_01_24_PavHidalgo.xlsm
Index(['Clave', 'Concepto', 'Unidad', 'Cantidad Contratada',
       'Precio Unitario/Costo', 'Importe Contratado', 'Est 1', 'Importe',
       'Est 2', 'Importe.1', 'Est 3', 'Importe.2', 'Est 4', 'Importe.3',
       'Est 5', 'Importe.4', 'Cantidad total estimada',
       'Importe total estimado', 'Marca', 'NIVEL'],
      dtype='object')
Se encontraron 27 conceptos con un porcentaje de 30.0% superior respecto del porcentaje contratado


,Clave,Partida_Principal,Subpartida,Concepto,Monto_Contratado,Monto_Ejecutado,Variacion_Pct_%
2,1.01,PRELIMINARES,N/A,Trazo y nivelación con equipo topográfic...,8085.50,11100.10,37.28%
3,1.02,PRELIMINARES,N/A,"Excavación con equipo en caja, en materi...",22566.92,30981.04,37.29%
4,1.03,PRELIMINARES,N/A,Acarreo en camión de material producto d...,35528.58,48775.53,37.29%
5,1.04,PRELIMINARES,N/A,Acarreo en camión de material producto d...,8733.45,11989.75,37.29%
10,2.01,TERRACERIAS,N/A,Compactación de terreno natural al 90% d...,20123.52,27626.36,37.28%
11,2.02,TERRACERIAS,N/A,SR-Relleno mecánico en caja con material...,101374.22,139155.07,37.27%
12,2.03,TERRACERIAS,N/A,SB-Relleno mecánico en caja con material...,91644.13,125812.65,37.28%
14,3.01,DRENAJE,N/A,Trazo y nivelación con equipo topográfic...,1135.25,1644.92,44.89%
15,3.02,DRENAJE,N/A,Excavación con equipo en zanja en materi...,4210.76,8826.12,109.61%
16,3.03,DRENAJE,N/A,Acarreo en camión de material producto d...,5966.51,12506.82,109.62%


--- RESUMEN DE AUDITORÍA POR SUBPARTIDAS ---


,Partida_Principal,Subpartida,Monto_Contratado,Monto_Ejecutado,Diferencia_Absoluta,%_Variacion_Global
4,PAVIMENTO,N/A,"$359,926.86","$521,641.70","$161,714.84",44.93%
2,GUARNICIONES Y BANQUETA,N/A,"$277,480.75","$396,760.30","$119,279.55",42.99%
6,TERRACERIAS,N/A,"$213,141.87","$292,594.08","$79,452.21",37.28%
1,DRENAJE,N/A,"$262,362.05","$295,076.79","$32,714.74",12.47%
5,PRELIMINARES,N/A,"$81,180.13","$109,112.10","$27,931.97",34.41%
0,AGUA POTABLE,N/A,"$82,784.73","$92,268.90","$9,484.17",11.46%
3,OBRA COMPLEMENTARIA,N/A,"$14,809.93","$20,149.12","$5,339.19",36.05%
7,FUERA DE CATALOGO,N/A,"$38,534.26","$38,534.26",$0.00,0.00%



--- ESTRATEGIA DE INSPECCIÓN FÍSICA (ANÁLISIS DE PARETO 80/20) ---
Total de conceptos en la obra: 52
Conceptos críticos a revisar para cubrir el 80% del monto: 12
--------------------------------------------------


,Partida_Principal,Subpartida,Clave,Concepto,Cantidad total estimada,Monto_Ejecutado,%_Peso,%_Acumulado
48,PAVIMENTO,N/A,6.010000,"Losa de concreto hidráulico con MR = 42 kg/cm². Incluye: colado, vibrado, curado, retiro de la cimbra y limpieza. N.CTR.CAR.1.04.009/20. (15.00 cm de espesor)",124,"$482,291.38",27.31%,27.31%
46,GUARNICIONES Y BANQUETA,N/A,5.060000,"Banqueta en la orilla del pavimento con ancho variable m y espesor de 10 cm de concreto hidráulico normal de f´c=150 kg/cm2 colada en el lugar por unidad de obra terminada, incluye: compactación y compensación de la superficie de desplante con material calidad subrasante, cimbras en fronteras, armado, colado, vibrado, curado, retiro de cimbra y limpieza. N-CTR-CAR-1-02-010/00.",538,"$211,982.85",12.00%,39.31%
11,TERRACERIAS,N/A,2.020000,"SR-Relleno mecánico en caja con material calidad de SUBRASANTE, compactado al 95% de su PVSM en capas de 20 cm. Incluye: abundamiento, acarreo de material e incorporación de agua para compactar, de acuerdo a norma N-CTR-CAR-1-01-011/20.",248,"$139,155.07",7.88%,47.19%
12,TERRACERIAS,N/A,2.030000,"SB-Relleno mecánico en caja con material calidad de sub-base, compactado al 95% de su PVSM en capas de 20 cm. Incluye: abundamiento, acarreo de material e incorporación de agua para compactar, de acuerdo a norma N-CTR-CAR-1-01-011/20.",165,"$125,812.65",7.12%,54.31%
41,GUARNICIONES Y BANQUETA,N/A,5.010000,"Guarnición trapezoidal de 15 x 20 x 40 cm elaborada a base de concreto premezclado f'c = 200 kg/cm2, tma 19 mm (3/4""), acabado aparente. Incluye: trazo, cimbra en fronteras, colado, vibrado, curado, retiro de cimbra y limpieza. N·CTR·CAR·1·02·010/00",271,"$105,923.35",6.00%,60.31%
22,DRENAJE,N/A,3.090000,"Relleno semi-mecánico en zanja con material inerte, compactado al 90% de su PVSM en capas de 20 cm. Incluye: abundamiento, acarreo de material e incorporación de agua para compactar.",117,"$65,624.12",3.72%,64.03%
45,GUARNICIONES Y BANQUETA,N/A,5.050000,"Relleno semi-mecánico en zanja con material inerte, compactado al 90% de su PVSM en capas de 20 cm. Incluye: abundamiento, acarreo de material e incorporación de agua para compactar.",97,"$54,438.38",3.08%,67.11%
19,DRENAJE,N/A,3.060000,"Suministro de tubería de PVC alcantarillado sanitario serie 20, de 300 mm de diámetro nominal, de acuerdo a la NOM-001-CONGUA-2011 y NMX-E-215/1-CNCP-2012; incluye: flete, acarreos y maniobras locales puesto L.A.B. en el almacén de la obra de la localidad, anillo empaque en parte proporcional.",95,"$51,167.00",2.90%,70.01%
4,PRELIMINARES,N/A,1.030000,"Acarreo en camión de material producto de excavación al 1er km. Incluye: carga con equipo, transporte y descarga.",698,"$48,775.53",2.76%,72.77%
24,DRENAJE,N/A,3.110000,"Pozo de visita tronco cónico de 1.51 a 1.75 m de profundidad, construido con cimentación a base de losa de concreto simple de 15 cm de espesor para desplante con resistencia de f`c=200 kg/cm2, muro de tabique rojo recocido de 28 cm de espesor, asentado con mortero de cemento-arena proporción 1:3 adicionado con impermeabilizante integral en proporción de 2 kg/bulto de cemento, aplanados con el mismo mortero de 2.0 cm de espesor mínimo, terminado fino exterior y pulido interior, medias cañas de concreto, escalones de varilla corrugada de acero No 8 @ 40 cm. Sin brocal y tapa. Incluye: trazo, nivelación, excavación, afine y compactación de terreno natural, cimentación, construcción del pozo, relleno calidad subrasante compactado al 95% Proctor, carga y acarreo de material sobrante a al sitio de desperdicio.",3,"$43,711.38",2.47%,75.24%



--- RESUMEN COMPLETO DE PRIORIDADES ---


,Prioridad,Partida_Principal,Subpartida,Clave,Concepto,Cantidad total estimada,Monto_Ejecutado,%_Peso,%_Acumulado
48,ALTA (Grupo A - cubre el 80%),PAVIMENTO,N/A,6.010000,"Losa de concreto hidráulico con MR = 42 kg/cm². Incluye: colado, vibrado, curado, retiro de la cimbra y limpieza. N.CTR.CAR.1.04.009/20. (15.00 cm de espesor)",124,"$482,291.38",27.31%,27.31%
46,ALTA (Grupo A - cubre el 80%),GUARNICIONES Y BANQUETA,N/A,5.060000,"Banqueta en la orilla del pavimento con ancho variable m y espesor de 10 cm de concreto hidráulico normal de f´c=150 kg/cm2 colada en el lugar por unidad de obra terminada, incluye: compactación y compensación de la superficie de desplante con material calidad subrasante, cimbras en fronteras, armado, colado, vibrado, curado, retiro de cimbra y limpieza. N-CTR-CAR-1-02-010/00.",538,"$211,982.85",12.00%,39.31%
11,ALTA (Grupo A - cubre el 80%),TERRACERIAS,N/A,2.020000,"SR-Relleno mecánico en caja con material calidad de SUBRASANTE, compactado al 95% de su PVSM en capas de 20 cm. Incluye: abundamiento, acarreo de material e incorporación de agua para compactar, de acuerdo a norma N-CTR-CAR-1-01-011/20.",248,"$139,155.07",7.88%,47.19%
12,ALTA (Grupo A - cubre el 80%),TERRACERIAS,N/A,2.030000,"SB-Relleno mecánico en caja con material calidad de sub-base, compactado al 95% de su PVSM en capas de 20 cm. Incluye: abundamiento, acarreo de material e incorporación de agua para compactar, de acuerdo a norma N-CTR-CAR-1-01-011/20.",165,"$125,812.65",7.12%,54.31%
41,ALTA (Grupo A - cubre el 80%),GUARNICIONES Y BANQUETA,N/A,5.010000,"Guarnición trapezoidal de 15 x 20 x 40 cm elaborada a base de concreto premezclado f'c = 200 kg/cm2, tma 19 mm (3/4""), acabado aparente. Incluye: trazo, cimbra en fronteras, colado, vibrado, curado, retiro de cimbra y limpieza. N·CTR·CAR·1·02·010/00",271,"$105,923.35",6.00%,60.31%
22,ALTA (Grupo A - cubre el 80%),DRENAJE,N/A,3.090000,"Relleno semi-mecánico en zanja con material inerte, compactado al 90% de su PVSM en capas de 20 cm. Incluye: abundamiento, acarreo de material e incorporación de agua para compactar.",117,"$65,624.12",3.72%,64.03%
45,ALTA (Grupo A - cubre el 80%),GUARNICIONES Y BANQUETA,N/A,5.050000,"Relleno semi-mecánico en zanja con material inerte, compactado al 90% de su PVSM en capas de 20 cm. Incluye: abundamiento, acarreo de material e incorporación de agua para compactar.",97,"$54,438.38",3.08%,67.11%
19,ALTA (Grupo A - cubre el 80%),DRENAJE,N/A,3.060000,"Suministro de tubería de PVC alcantarillado sanitario serie 20, de 300 mm de diámetro nominal, de acuerdo a la NOM-001-CONGUA-2011 y NMX-E-215/1-CNCP-2012; incluye: flete, acarreos y maniobras locales puesto L.A.B. en el almacén de la obra de la localidad, anillo empaque en parte proporcional.",95,"$51,167.00",2.90%,70.01%
4,ALTA (Grupo A - cubre el 80%),PRELIMINARES,N/A,1.030000,"Acarreo en camión de material producto de excavación al 1er km. Incluye: carga con equipo, transporte y descarga.",698,"$48,775.53",2.76%,72.77%
24,ALTA (Grupo A - cubre el 80%),DRENAJE,N/A,3.110000,"Pozo de visita tronco cónico de 1.51 a 1.75 m de profundidad, construido con cimentación a base de losa de concreto simple de 15 cm de espesor para desplante con resistencia de f`c=200 kg/cm2, muro de tabique rojo recocido de 28 cm de espesor, asentado con mortero de cemento-arena proporción 1:3 adicionado con impermeabilizante integral en proporción de 2 kg/bulto de cemento, aplanados con el mismo mortero de 2.0 cm de espesor mínimo, terminado fino exterior y pulido interior, medias cañas de concreto, escalones de varilla corrugada de acero No 8 @ 40 cm. Sin brocal y tapa. Incluye: trazo, nivelación, excavación, afine y compactación de terreno natural, cimentación, construcción del pozo, relleno calidad subrasante compactado al 95% Proctor, carga y acarreo de material sobrante a al sitio de desperdicio.",3,"$43,711.38",2.47%,75.24%


In [ ]:
!pip install xlsxwriter
output_excel_filename = "Resultados_Auditoria.xlsx"

with pd.ExcelWriter(output_excel_filename, engine='xlsxwriter') as writer:
    # Exportar el análisis de excesos
    if not excesos.empty:
        excesos.to_excel(writer, sheet_name='Conceptos_con_Exceso', index=False)
    else:
        pd.DataFrame(["No se encontraron conceptos con exceso."]).to_excel(writer, sheet_name='Conceptos_con_Exceso', index=False, header=False)

    # Exportar el resumen ejecutivo
    if not resumen_ejecutivo.empty:
        # Para mantener la visibilidad de los datos formateados, convertimos las columnas a string si ya tienen formato aplicado
        # o exportamos los valores numéricos directos si se quiere la capacidad de cálculo en Excel.
        # Para este ejemplo, exportaremos los valores numéricos puros para que se puedan manipular en Excel.
        resumen_ejecutivo_export = resumen_ejecutivo.copy()
        # Las columnas Monto_Contratado, Monto_Ejecutado, Diferencia_Absoluta y %_Variacion_Global ya son numéricas.
        # No es necesario convertirlas de string a float.
        resumen_ejecutivo_export.to_excel(writer, sheet_name='Resumen_Ejecutivo', index=False)
    else:
        pd.DataFrame(["No hay resumen ejecutivo disponible."]).to_excel(writer, sheet_name='Resumen_Ejecutivo', index=False, header=False)

    # Exportar la lista de campo (prioridad ALTA)
    if not lista_campo.empty:
        lista_campo.to_excel(writer, sheet_name='Inspeccion_Campo_ALTA', index=False)
    else:
        pd.DataFrame(["No se encontraron conceptos de alta prioridad para inspección."]).to_excel(writer, sheet_name='Inspeccion_Campo_ALTA', index=False, header=False)

    # Exportar el resumen completo de prioridades
    if not df_plan_inspeccion_filtrado.empty:
        df_plan_inspeccion_filtrado.to_excel(writer, sheet_name='Resumen_Prioridades_Completo', index=False)
    else:
        pd.DataFrame(["No hay resumen completo de prioridades disponible."]).to_excel(writer, sheet_name='Resumen_Prioridades_Completo', index=False, header=False)

print(f"Todos los análisis han sido exportados a '{output_excel_filename}' en diferentes hojas.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 11.1 MB/s eta 0:00:00
Todos los análisis han sido exportados a 'Resultados_Auditoria.xlsx' en diferentes hojas.
